In [1]:
!pip install transformers datasets peft accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 10.5 MB/s eta 0:00:00


Training script for fine-tuning Phi-3.5-mini for function calling
Uses synthetic data generation and LoRA for efficient training



In [ ]:
import json
import random
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
    BitsAndBytesConfig
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
import torch

# ============================================================================
# STEP 1: Generate Synthetic Function Calling Data
# ============================================================================

def generate_synthetic_data(num_samples=2000):
    """Generate synthetic function calling examples"""

    # Define available functions with descriptions
    functions = {
        "get_weather": {
            "params": ["location", "unit"],
            "templates": [
                "What's the weather in {location}?",
                "Tell me the weather for {location}",
                "How's the weather in {location} today?",
                "Get weather for {location} in {unit}",
                "Weather forecast for {location}"
            ]
        },
        "search_database": {
            "params": ["query", "table", "limit"],
            "templates": [
                "Search for {query} in {table}",
                "Find {query} in the {table} table",
                "Look up {query} limited to {limit} results",
                "Query the {table} for {query}"
            ]
        },
        "send_email": {
            "params": ["recipient", "subject", "body"],
            "templates": [
                "Send email to {recipient} with subject {subject}",
                "Email {recipient} about {subject}: {body}",
                "Send {recipient} an email titled {subject}",
                "Compose email to {recipient} with subject {subject} and message {body}"
            ]
        },
        "calculate": {
            "params": ["expression"],
            "templates": [
                "Calculate {expression}",
                "What is {expression}?",
                "Compute {expression}",
                "Solve {expression}"
            ]
        },
        "set_reminder": {
            "params": ["task", "time", "date"],
            "templates": [
                "Remind me to {task} at {time} on {date}",
                "Set reminder for {task} at {time}",
                "Create reminder: {task} on {date} at {time}",
                "Add reminder to {task} at {time}"
            ]
        },
        "get_stock_price": {
            "params": ["symbol"],
            "templates": [
                "What's the stock price of {symbol}?",
                "Get stock price for {symbol}",
                "Check {symbol} stock price",
                "Show me {symbol} current price"
            ]
        },
        "translate": {
            "params": ["text", "source_lang", "target_lang"],
            "templates": [
                "Translate {text} from {source_lang} to {target_lang}",
                "Convert {text} to {target_lang}",
                "Translate '{text}' into {target_lang}"
            ]
        },
        "book_appointment": {
            "params": ["service", "date", "time"],
            "templates": [
                "Book {service} appointment on {date} at {time}",
                "Schedule {service} for {date} at {time}",
                "Make appointment for {service} on {date}"
            ]
        }
    }

    # Sample data for parameters
    sample_values = {
        "location": ["New York", "London", "Tokyo", "Paris", "Sydney", "Berlin", "Toronto"],
        "unit": ["celsius", "fahrenheit"],
        "query": ["recent orders", "customer data", "sales records", "inventory items"],
        "table": ["customers", "orders", "products", "employees"],
        "limit": ["10", "20", "50", "100"],
        "recipient": ["john@example.com", "sarah@company.com", "team@startup.io"],
        "subject": ["Meeting Update", "Project Status", "Weekly Report", "Quarterly Review"],
        "body": ["Please review the attached document", "Looking forward to our discussion", "Here are the updates"],
        "expression": ["2 + 2", "15 * 8", "100 / 4", "sqrt(144)", "(5 + 3) * 2"],
        "task": ["buy groceries", "call the dentist", "submit report", "review code"],
        "time": ["10:00 AM", "2:30 PM", "9:00 AM", "5:00 PM"],
        "date": ["2024-03-15", "2024-03-20", "tomorrow", "next Monday"],
        "symbol": ["AAPL", "GOOGL", "MSFT", "TSLA", "AMZN"],
        "text": ["Hello, how are you?", "Good morning", "Thank you very much"],
        "source_lang": ["English", "Spanish", "French", "German"],
        "target_lang": ["Spanish", "French", "German", "Japanese"]
    }

    dataset = []

    for _ in range(num_samples):
        # Randomly select a function
        func_name = random.choice(list(functions.keys()))
        func_info = functions[func_name]

        # Select a template
        template = random.choice(func_info["templates"])

        # Generate parameters
        params = {}
        query = template

        for param in func_info["params"]:
            if param in sample_values:
                value = random.choice(sample_values[param])
                params[param] = value
                query = query.replace(f"{{{param}}}", value)

        # Create function call JSON
        function_call = {
            "function": func_name,
            "parameters": params
        }

        # Format as training example
        example = {
            "messages": [
                {
                    "role": "user",
                    "content": query
                },
                {
                    "role": "assistant",
                    "content": json.dumps(function_call)
                }
            ]
        }

        dataset.append(example)

    return dataset

# ============================================================================
# STEP 2: Prepare Dataset for Training
# ============================================================================

def format_chat_template(example, tokenizer):
    """Format examples using the model's chat template"""

    # Apply chat template
    formatted = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False
    )

    return {"text": formatted}

def prepare_dataset(synthetic_data, tokenizer):
    """Convert synthetic data to Hugging Face Dataset"""

    # Create dataset
    dataset = Dataset.from_list(synthetic_data)

    # Apply chat template formatting
    dataset = dataset.map(
        lambda x: format_chat_template(x, tokenizer),
        remove_columns=dataset.column_names
    )

    # Tokenize
    def tokenize_function(examples):
        return tokenizer(
            examples["text"],
            truncation=True,
            max_length=512,
            padding=False
        )

    tokenized_dataset = dataset.map(
        tokenize_function,
        batched=True,
        remove_columns=dataset.column_names
    )

    return tokenized_dataset

# ============================================================================
# STEP 3: Training Setup
# ============================================================================

# Configuration
MODEL_NAME = "microsoft/Phi-3.5-mini-instruct"
OUTPUT_DIR = "./phi3-function-calling"
NUM_SAMPLES = 2000  # Small dataset - Phi-3.5 works well with limited data

print("=" * 80)
print("Training Phi-3.5-mini for Function Calling")
print("=" * 80)

# Generate synthetic data
print(f"\n[1/5] Generating {NUM_SAMPLES} synthetic training examples...")
synthetic_data = generate_synthetic_data(num_samples=NUM_SAMPLES)
print(f"✓ Generated {len(synthetic_data)} examples")

# Sample output
print("\nSample training example:")
print(json.dumps(synthetic_data[0], indent=2))

# Load tokenizer
print(f"\n[2/5] Loading tokenizer from {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"
print("✓ Tokenizer loaded")

# Prepare dataset
print("\n[3/5] Preparing dataset...")
train_dataset = prepare_dataset(synthetic_data, tokenizer)
print(f"✓ Dataset prepared: {len(train_dataset)} examples")

# Load model
print(f"\n[4/5] Loading model {MODEL_NAME}...")
# 4-bit quantization config (QLoRA) — loads model in ~2-3GB instead of ~8GB
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True  # nested quantization saves ~0.4GB extra
)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

# Prepare model for LoRA training
model = prepare_model_for_kbit_training(model)

# Configure LoRA
lora_config = LoraConfig(
    r=16,  # Rank
    lora_alpha=32,  # Scaling factor
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# Training arguments — tuned for ~15GB VRAM
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=8,        # reduced from 4
    gradient_accumulation_steps=8,        # keeps effective batch size at 16
    learning_rate=2e-4,
    fp16=False,
    bf16=True,
    logging_steps=10,
    save_strategy="epoch",
    warmup_steps=100,
    optim="paged_adamw_8bit",             # 8-bit optimizer lives on CPU, saves ~1GB
    gradient_checkpointing=True,          # recomputes activations instead of storing them
    report_to=None,
)

# Data collator
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

# Initialize trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=data_collator,
)

# Train
print("\n[5/5] Starting training...")
print("=" * 80)
trainer.train()

# Save model
print("\n" + "=" * 80)
print("Saving model...")
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"✓ Model saved to {OUTPUT_DIR}")

print("\n" + "=" * 80)
print("Training Complete!")
print("=" * 80)
print(f"\nYour model is saved in: {OUTPUT_DIR}")
print("\nTo use your model:")
print(f"  from transformers import AutoTokenizer, AutoModelForCausalLM")
print(f"  tokenizer = AutoTokenizer.from_pretrained('{OUTPUT_DIR}')")
print(f"  model = AutoModelForCausalLM.from_pretrained('{OUTPUT_DIR}')")


Training Phi-3.5-mini for Function Calling

[1/5] Generating 2000 synthetic training examples...
✓ Generated 2000 examples

Sample training example:
{
  "messages": [
    {
      "role": "user",
      "content": "Search for sales records in orders"
    },
    {
      "role": "assistant",
      "content": "{\"function\": \"search_database\", \"parameters\": {\"query\": \"sales records\", \"table\": \"orders\", \"limit\": \"20\"}}"
    }
  ]
}

[2/5] Loading tokenizer from microsoft/Phi-3.5-mini-instruct...
✓ Tokenizer loaded

[3/5] Preparing dataset...


Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

✓ Dataset prepared: 2000 examples

[4/5] Loading model microsoft/Phi-3.5-mini-instruct...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

trainable params: 8,912,896 || all params: 3,829,992,448 || trainable%: 0.2327

[5/5] Starting training...


Step,Training Loss
10,2.504700
20,2.296300
30,1.711300
40,1.124700
50,0.779100
60,0.525000
70,0.327500
80,0.208700
90,0.171300



Saving model...
✓ Model saved to ./phi3-function-calling

Training Complete!

Your model is saved in: ./phi3-function-calling

To use your model:
  from transformers import AutoTokenizer, AutoModelForCausalLM
  tokenizer = AutoTokenizer.from_pretrained('./phi3-function-calling')
  model = AutoModelForCausalLM.from_pretrained('./phi3-function-calling')


**Test script for the trained function calling model**

In [20]:
import gc, torch
gc.collect()
torch.cuda.empty_cache()

"""
Test script for the trained function calling model
"""

"""
Test script for the trained function calling model
"""

import json
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

BASE_MODEL = "microsoft/Phi-3.5-mini-instruct"

def load_model(model_path="./phi3-function-calling"):
    """Load base model in 4-bit, then attach the LoRA adapter on top"""
    print(f"Loading base model in 4-bit...")

    # Must match the quantization used during training
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True
    )

    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
    tokenizer.pad_token = tokenizer.eos_token

    # Step 1: load the original base model quantized
    model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True
    )

    # Step 2: attach your trained LoRA weights on top
    print(f"Attaching LoRA adapter from {model_path}...")
    model = PeftModel.from_pretrained(model, model_path)

    print("✓ Model loaded successfully")
    return tokenizer, model

def predict_function_call(query, tokenizer, model):
    """Generate function call for a given query"""

    # Format as chat
    messages = [
        {"role": "user", "content": query}
    ]

    # Apply chat template
    input_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    # Tokenize — use cuda explicitly; quantized models can report 'meta' as device
    device = "cuda" if torch.cuda.is_available() else "cpu"
    inputs = tokenizer(input_text, return_tensors="pt").to(device)

    # Generate
    # use_cache=False avoids a bug in Phi-3.5's cached modeling_phi3.py
    # where DynamicCache.seen_tokens was renamed to _seen_tokens in newer transformers
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=150,
            temperature=0.1,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
            use_cache=False
        )

    # Decode
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Extract the JSON function call.
    # The model echoes the user query before outputting the JSON, so we
    # find the first '{' and extract everything from there to the last '}'.
    start = response.find("{")
    end = response.rfind("}")
    if start != -1 and end != -1:
        response = response[start:end + 1]

    return response

print("=" * 80)
print("Function Calling Model - Interactive Testing")
print("=" * 80)

# Load model
tokenizer, model = load_model()

# Test queries
test_queries = [
    "What's the weather in Tokyo?",
    "Send an email to john@example.com about the meeting",
    "Remind me to buy groceries at 5:00 PM tomorrow",
    "Calculate 15 * 8",
    "Get the stock price of AAPL",
    "Translate 'Hello, how are you?' from English to Spanish",
    "Search for recent orders in the customers table",
    "Book a haircut appointment on 2024-03-20 at 2:30 PM"
]

print("\n" + "=" * 80)
print("Running Test Queries")
print("=" * 80 + "\n")

for i, query in enumerate(test_queries, 1):
    print(f"[{i}/{len(test_queries)}] Query: {query}")

    result = predict_function_call(query, tokenizer, model)
    print(f"Response: {result}")

    # Try to parse as JSON to verify format
    try:
        parsed = json.loads(result)
        print(f"✓ Valid JSON - Function: {parsed.get('function', 'N/A')}")
    except:
        print("⚠ Not valid JSON format")

    print("-" * 80 + "\n")

# Interactive mode
print("\n" + "=" * 80)
print("Interactive Mode (type 'quit' to exit)")
print("=" * 80 + "\n")

while True:
    query = input("Enter query: ").strip()

    if query.lower() in ['quit', 'exit', 'q']:
        break

    if not query:
        continue

    result = predict_function_call(query, tokenizer, model)
    print(f"\nResponse: {result}\n")

    try:
        parsed = json.loads(result)
        print("Parsed function call:")
        print(json.dumps(parsed, indent=2))
    except:
        print("(Could not parse as JSON)")

    print()



Function Calling Model - Interactive Testing
Loading base model in 4-bit...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Attaching LoRA adapter from ./phi3-function-calling...
✓ Model loaded successfully

Running Test Queries

[1/8] Query: What's the weather in Tokyo?
Response: {"function": "get_weather", "parameters": {"location": "Tokyo", "unit": "fahrenheit"}}
✓ Valid JSON - Function: get_weather
--------------------------------------------------------------------------------

[2/8] Query: Send an email to john@example.com about the meeting


KeyboardInterrupt: 